In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from sklearn.datasets import load_iris
from ml_tooling import Model
from ml_tooling.data import Dataset
from ml_tooling.baseclass import logger
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
class IrisData(Dataset):
    def load_training_data(self):
        data = load_iris()
        return pd.DataFrame(data=data.data, columns=data.feature_names), np.where(data.target == 1, 1, 0)
    
    def load_prediction_data(self, idx):
        X, y = self.load_training_data()
        
        return X.loc[idx, :].to_frame().T
    
data = IrisData()
data.create_train_test()

In [ ]:
rf_clf = Model(RandomForestClassifier(n_estimators=10))

In [ ]:
lr = Model(LogisticRegression())

In [ ]:
lr.score_estimator(data)

In [ ]:
rf_clf.config.RANDOM_STATE = 42

In [ ]:
rf_clf.config

In [ ]:
result = rf_clf.score_estimator(data, metrics='roc_auc', cv=10)

In [ ]:
result.plot.lift_curve()

In [ ]:
ax = result.plot.feature_importance(100)

In [ ]:
with rf_clf.log('./gridsearch'):
    best_model, results = rf_clf.gridsearch(data, {"max_depth": [5, 10, 15]})

In [ ]:
best_model

In [ ]:
results

In [ ]:
with rf_clf.log('./test'):
    rf_clf.score_estimator(data)

In [ ]:
from ml_tooling.storage import FileStorage

storage = FileStorage('./save')
with rf_clf.log('./save'):
    estimator_path = rf_clf.save_estimator(storage)

In [ ]:
test_clf = Model.load_estimator(storage, estimator_path.name)

In [ ]:
test_clf.estimator

In [ ]:
test_clf.make_prediction(data, 19, proba=True)

In [ ]:
from sklearn.dummy import DummyClassifier

In [ ]:
best_model, results = Model.test_estimators(data, [RandomForestClassifier(n_estimators=10), LogisticRegression(), DummyClassifier()], log_dir='')

In [ ]:
results